# Thesis PDF processing

1. **Trimmed PDFs** — drop pages from **References / appendix** onward (`thesisdatarepo.pdf_context`).
2. **NLP text** — UTF-8 **`.txt`** from **Abstract** (or page 1 if not found) through the last page **before** back matter, plus manifest CSV/JSON and optional **`corpus.jsonl`** (`thesisdatarepo.nlp_extract`).

**Environment:** from the repo root, use uv so the package is on the path:

```bash
uv run jupyter lab maks/Analysis_main.ipynb
```

Or: `uv sync` then select the project interpreter (`.venv`).

**Data:** either put PDFs in a local folder, or **stream from GCS** with `process_gcs_prefix_nlp` (no local `.pdf` files; uses `google-cloud-storage` + the same ADC as `gcloud`).

Local copy (optional):

```bash
mkdir -p maks/data/master_thesis_sample
gsutil -m cp "gs://thesis_archive_bucket/dtu_findit/master_thesis/*.pdf" maks/data/master_thesis_sample/  # large; prefer a subset
```


In [ ]:
from pathlib import Path

from thesisdatarepo.pdf_context import FallbackPolicy, process_folder, process_one_pdf

# Prefer repo root whether the kernel cwd is `maks/` or the project root.
_here = Path.cwd().resolve()
REPO_ROOT = _here if (_here / "pyproject.toml").is_file() else (_here / "..").resolve()
INPUT_DIR = REPO_ROOT / "maks" / "data" / "master_thesis_sample"
OUTPUT_DIR = REPO_ROOT / "maks" / "data" / "master_thesis_context"
MANIFEST_CSV = REPO_ROOT / "maks" / "data" / "manifest_context.csv"
MANIFEST_JSON = REPO_ROOT / "maks" / "data" / "manifest_context.json"

INPUT_DIR, OUTPUT_DIR

In [ ]:
# Batch: all PDFs in INPUT_DIR -> OUTPUT_DIR + manifest (CSV)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

results = process_folder(
    INPUT_DIR,
    OUTPUT_DIR,
    MANIFEST_CSV,
    manifest_format="csv",
    min_page_fraction=0.45,
    policy=FallbackPolicy.KEEP_ALL,
)
len(results), MANIFEST_CSV

In [ ]:
# Optional: JSON manifest
from thesisdatarepo.pdf_context import process_folder

_ = process_folder(
    INPUT_DIR,
    OUTPUT_DIR,
    MANIFEST_JSON,
    manifest_format="json",
    min_page_fraction=0.45,
)


In [ ]:
# Single-file example (first PDF in folder)
pdfs = sorted(INPUT_DIR.glob("*.pdf"))
if pdfs:
    out = OUTPUT_DIR / "_single_sample.pdf"
    r = process_one_pdf(
        pdfs[0],
        out,
        min_page_fraction=0.45,
        policy=FallbackPolicy.KEEP_ALL,
    )
    r
else:
    print("No PDFs in", INPUT_DIR)

### NLP export (Abstract → before references)

Writes `maks/data/nlp_txt/<name>.txt`, `maks/data/manifest_nlp.csv`, and `maks/data/corpus.jsonl` (one JSON object per line: `id`, `text`, `meta`).

In [ ]:
from thesisdatarepo.nlp_extract import process_folder_nlp
from thesisdatarepo.pdf_context import FallbackPolicy

NLP_TXT_DIR = REPO_ROOT / "maks" / "data" / "nlp_txt"
MANIFEST_NLP = REPO_ROOT / "maks" / "data" / "manifest_nlp.csv"
CORPUS_JSONL = REPO_ROOT / "maks" / "data" / "corpus.jsonl"

NLP_TXT_DIR, MANIFEST_NLP, CORPUS_JSONL

In [ ]:
INPUT_DIR.mkdir(parents=True, exist_ok=True)

nlp_results = process_folder_nlp(
    INPUT_DIR,
    NLP_TXT_DIR,
    MANIFEST_NLP,
    manifest_format="csv",
    jsonl_path=CORPUS_JSONL,
    min_page_fraction_back_matter=0.45,
    policy=FallbackPolicy.KEEP_ALL,
)
len(nlp_results), MANIFEST_NLP, CORPUS_JSONL

### NLP from GCS (stream PDFs in memory)

PDFs are read from the bucket into memory; only **`.txt`**, manifest, and **`corpus.jsonl`** are written under `maks/data/`. Use **`max_workers`** to parallelize. **`resume=True`** (default) skips blobs already recorded in **`nlp_from_gcs/.nlp_checkpoints.jsonl`** so you can stop and continue; **`download_failed`** is not checkpointed (retries). Use **`force_fresh=True`** to ignore checkpoints and start clean. Set **`max_objects`** or `None` for the whole prefix.

In [ ]:
from thesisdatarepo.gcs_nlp import process_gcs_prefix_nlp

GCS_BUCKET = "thesis_archive_bucket"
GCS_PREFIX = "dtu_findit/master_thesis"
NLP_FROM_GCS_DIR = REPO_ROOT / "maks" / "data" / "nlp_from_gcs"
MANIFEST_NLP_GCS = REPO_ROOT / "maks" / "data" / "manifest_nlp_gcs.csv"
CORPUS_GCS_JSONL = REPO_ROOT / "maks" / "data" / "corpus_gcs.jsonl"

gcs_results = process_gcs_prefix_nlp(
    GCS_BUCKET,
    GCS_PREFIX,
    NLP_FROM_GCS_DIR,
    MANIFEST_NLP_GCS,
    max_objects=10,
    max_workers=8,
    jsonl_path=CORPUS_GCS_JSONL,
    policy=FallbackPolicy.KEEP_ALL,
)
len(gcs_results), MANIFEST_NLP_GCS